# RBA Login Dataset — Risk-Based Authentication (User Behaviour)

| | |
|---|---|
| **Source** | Kaggle `dasgroup/rba-dataset` |
| **Origin** | Wiefling et al., "Pump Up Password Security!" (ACM TOPS 2022) — real SSO logins from a large online service, anonymized |
| **License** | CC BY-NC 4.0 |
| **Samples** | ~33M login events over ~1 year |
| **Features** | timestamp, user ID, IP, country/region/city, ASN, user agent, browser/OS, device type, RTT, login success |
| **Labels** | `Is Attack IP`, `Is Account Takeover` |
| **Caveats** | Extreme size → stratified sample: ALL attack/ATO rows + benign sample (documented). NC license. |
| **Production research suitability** | HIGH — real behavioural login telemetry with geo + device: ideal for impossible-travel features. |

In [1]:
import sys, glob
sys.path.insert(0, "..")
import numpy as np
import pandas as pd
from prep_utils import RAW, to_unified, dataset_report, numeric_summary, save_clean, save_unified_part

D = RAW / "behaviour" / "rba"
BENIGN_SAMPLE_FRAC = 0.06  # ~2M benign rows from ~33M — documented sampling

## Chunked load with stratified sampling
Keep 100% of `Is Attack IP` / `Is Account Takeover` rows, sample benign at 6%.
Sampling is chunk-uniform with fixed seed → unbiased w.r.t. time.

In [2]:
from prep_utils import CLEAN
FAST_PATH = (CLEAN / "rba.parquet")
if FAST_PATH.exists():
    # idempotent fast path: sampling+cleaning already done in a prior run
    df = pd.read_parquet(FAST_PATH)
    print(f"loaded prior clean sample: {len(df):,} rows (delete {FAST_PATH} to rescan raw csv)")
else:
    csv = glob.glob(str(D / "*.csv"))[0]
    rng = np.random.default_rng(42)
    keep_parts, total = [], 0
    for chunk in pd.read_csv(csv, chunksize=1_000_000):
        total += len(chunk)
        atk = chunk[(chunk["Is Attack IP"]) | (chunk["Is Account Takeover"])]
        ben = chunk[~((chunk["Is Attack IP"]) | (chunk["Is Account Takeover"]))]
        ben = ben.sample(frac=BENIGN_SAMPLE_FRAC, random_state=42)
        keep_parts.append(pd.concat([atk, ben]))
    df = pd.concat(keep_parts, ignore_index=True)
    print(f"total rows scanned: {total:,}; kept: {len(df):,}")
df.head(3)

loaded prior clean sample: 4,787,377 rows (delete /home/suryaguru/StudioProjects/sentinel_fusion_ai/data/clean/rba.parquet to rescan raw csv)


,index,user_id,rtt_ms,ip_address,country,region,city,asn,browser_name_and_version,os_name_and_version,device_type,login_successful,is_attack_ip,is_account_takeover,rtt_ms_missing,event_time,label_bin
0,4,-4618854071942621186,542.0,10.0.0.47,US,Virginia,Ashburn,398986,Chrome Mobile WebView 85.0.4183,Android 2.2,mobile,False,True,False,1,2020-02-03 12:43:59.396000+00:00,1
1,5,-4324475583306591935,542.0,209.236.123.126,US,-,-,393398,Chrome Mobile WebView 85.0.4183,Android 4.1,mobile,False,True,False,1,2020-02-03 12:44:05.160000+00:00,1
2,9,-3065936140549856249,542.0,92.221.109.162,NO,Rogaland,Sandnes,29695,Chrome 69.0.3497.17.28,Mac OS X 10.14.6,desktop,True,False,False,1,2020-02-03 12:44:19.071000+00:00,0


## Cleaning

In [3]:
df.columns = [c.strip().lower().replace(" ", "_").replace("[ms]", "ms").replace("-", "_") for c in df.columns]
df = df.rename(columns={"round_trip_time_ms_": "rtt_ms", "round_trip_time_ms": "rtt_ms"})
rtt = [c for c in df.columns if "round" in c or "rtt" in c][0]
df = df.rename(columns={rtt: "rtt_ms"})
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"dropped {before - len(df)} duplicates")
print("missing:", {c: int(v) for c, v in df.isna().sum().items() if v})
if "rtt_ms_missing" not in df.columns:  # skip on fast path — already imputed
    df["rtt_ms_missing"] = df["rtt_ms"].isna().astype("int8")
    df["rtt_ms"] = df["rtt_ms"].fillna(df["rtt_ms"].median())
for c in ["country", "device_type", "browser_name_and_version", "os_name_and_version"]:
    if c in df.columns:
        df[c] = df[c].astype(str).fillna("unknown").astype("category")

dropped 0 duplicates
missing: {'region': 3219, 'city': 1054}


## Timestamp normalization — real timestamps here

In [4]:
if "login_timestamp" in df.columns:
    df["event_time"] = pd.to_datetime(df["login_timestamp"], utc=True, errors="coerce")
bad = int(df["event_time"].isna().sum())
print("unparseable timestamps:", bad)
df = df.dropna(subset=["event_time"]).sort_values("event_time").reset_index(drop=True)

unparseable timestamps: 0


## Label verification

In [5]:
df["label_bin"] = ((df["is_attack_ip"]) | (df["is_account_takeover"])).astype("int8")
print(pd.crosstab(df["is_attack_ip"], df["is_account_takeover"]))
df["label_bin"].value_counts(normalize=True)

is_account_takeover    False  True 
is_attack_ip                       
False                1690336     64
True                 3096900     77


label_bin
1    0.646918
0    0.353082
Name: proportion, dtype: float64

## EDA

In [6]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
df["country"].value_counts().head(12).plot.barh(ax=axes[0], title="top countries")
df.groupby(df["event_time"].dt.hour)["label_bin"].mean().plot(ax=axes[1], title="attack rate by hour (UTC)")
df["device_type"].value_counts().plot.bar(ax=axes[2], title="device types")
plt.tight_layout(); plt.savefig("../reports/rba_eda.png", dpi=110); plt.show()

/tmp/ipykernel_220382/1907844645.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig("../reports/rba_eda.png", dpi=110); plt.show()


In [7]:
numeric_summary(df, "rba")

,count,mean,std,min,25%,50%,75%,max
index,4787377.0,1.594647e+07,9.165551e+06,4.000000e+00,7.955496e+06,1.600933e+07,2.409855e+07,3.126925e+07
user_id,4787377.0,-2.163994e+18,4.345472e+18,-9.223361e+18,-4.324476e+18,-4.324476e+18,7.557359e+14,9.223308e+18
rtt_ms,4787377.0,5.449448e+02,1.777241e+02,9.000000e+00,5.420000e+02,5.420000e+02,5.420000e+02,1.018300e+05
asn,4787377.0,2.711518e+05,1.733509e+05,1.200000e+01,4.804300e+04,3.933980e+05,3.933980e+05,5.077260e+05
rtt_ms_missing,4787377.0,9.800970e-01,1.396669e-01,0.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00
label_bin,4787377.0,6.469181e-01,4.779279e-01,0.000000e+00,0.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00


## Save clean + unified

In [8]:
drop_cols = [c for c in ["login_timestamp", "user_agent_string"] if c in df.columns]
clean = df.drop(columns=drop_cols)
save_clean(clean, "rba")
dataset_report(clean, "rba", label_col="label_bin",
               notes=f"Stratified sample: all attack/ATO + {BENIGN_SAMPLE_FRAC:.0%} benign of ~33M rows. Real timestamps.")

clean -> /home/suryaguru/StudioProjects/sentinel_fusion_ai/data/clean/rba.parquet (4,787,377 rows)


report -> /home/suryaguru/StudioProjects/sentinel_fusion_ai/reports/rba_stats.json


{'dataset': 'rba',
 'rows': 4787377,
 'columns': 17,
 'memory_mb': 463.2,
 'duplicate_rows': 0,
 'missing_by_column': {'region': 3219, 'city': 1054},
 'dtypes': {'index': 'int64',
  'user_id': 'int64',
  'rtt_ms': 'float64',
  'ip_address': 'str',
  'country': 'category',
  'region': 'str',
  'city': 'str',
  'asn': 'int64',
  'browser_name_and_version': 'category',
  'os_name_and_version': 'category',
  'device_type': 'category',
  'login_successful': 'bool',
  'is_attack_ip': 'bool',
  'is_account_takeover': 'bool',
  'rtt_ms_missing': 'int8',
  'event_time': 'datetime64[us, UTC]',
  'label_bin': 'int8'},
 'notes': 'Stratified sample: all attack/ATO + 6% benign of ~33M rows. Real timestamps.',
 'label_distribution': {'1': 3097041, '0': 1690336},
 'imbalance_ratio': 1.83}

In [9]:
u = pd.DataFrame({
    "event_time": clean["event_time"],
    "event_subtype": np.where(clean["login_successful"], "login_success", "login_fail"),
    "user_id": clean["user_id"].astype(str),
    "device_id": clean["device_type"].astype(str),
    "src_ip": clean["ip_address"].astype(str),
    "country": clean["country"].astype(str),
    "duration_s": clean["rtt_ms"] / 1000.0,
    "severity": np.select([clean["is_account_takeover"], clean["is_attack_ip"]], [4, 3], 0).astype("int8"),
    "label": clean["label_bin"].astype("Int8"),
    "label_type": np.where(clean["is_account_takeover"], "account_takeover", "attack"),
    "time_is_synthetic": False,
})
attr_cols = ["region", "city", "asn", "browser_name_and_version", "os_name_and_version",
             "is_attack_ip", "is_account_takeover", "rtt_ms_missing"]
attr_cols = [c for c in attr_cols if c in clean.columns]
u[attr_cols] = clean[attr_cols]
u = to_unified(u, source_dataset="rba", event_domain="behaviour",
               event_type="login", label_type="account_takeover", attributes_cols=attr_cols)
save_unified_part(u, "rba")
u.head(3)

unified part -> /home/suryaguru/StudioProjects/sentinel_fusion_ai/data/unified/part_rba.parquet (4,787,377 rows)


,event_id,event_time,event_domain,event_type,event_subtype,source_dataset,user_id,device_id,src_ip,dst_ip,...,amount,duration_s,bytes_in,bytes_out,severity,label,label_type,attack_technique,time_is_synthetic,attributes
0,rba-0,2020-02-03 12:43:59.396000+00:00,behaviour,login,login_fail,rba,-4618854071942621186,mobile,10.0.0.47,<NA>,...,NaN,0.542,NaN,NaN,3,1,attack,<NA>,False,"{""region"":""Virginia"",""city"":""Ashburn"",""asn"":39..."
1,rba-1,2020-02-03 12:44:05.160000+00:00,behaviour,login,login_fail,rba,-4324475583306591935,mobile,209.236.123.126,<NA>,...,NaN,0.542,NaN,NaN,3,1,attack,<NA>,False,"{""region"":""-"",""city"":""-"",""asn"":393398,""browser..."
2,rba-2,2020-02-03 12:44:19.071000+00:00,behaviour,login,login_success,rba,-3065936140549856249,desktop,92.221.109.162,<NA>,...,NaN,0.542,NaN,NaN,0,0,attack,<NA>,False,"{""region"":""Rogaland"",""city"":""Sandnes"",""asn"":29..."
